# Notebook 3 — Brief Bayesian GP on short-horizon solar

**Interview appendix only.** ExactGP with RBF + periodic kernel on a few days of hourly PV output: posterior mean ± 2σ for a 12-hour horizon. One paragraph on why production desks prefer conformal CQR instead.

See also: [`docs/conformal_mental_model.md`](../docs/conformal_mental_model.md)

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import pandas as pd

from src.conformal.gp_solar import SolarSimulationConfig, fit_gp_solar, simulate_solar_output
from src.plotting import (
    COLOR_ACTUAL,
    COLOR_GP_MEAN,
    save_summary_figure,
    setup_style,
)

setup_style()

## Kernel choice

Short-horizon solar is smooth hour-to-hour (**RBF**) and repeats every 24 h (**periodic**, period init = 24). We use GPyTorch **ExactGP** with an additive kernel — fine for ~80 training hours; O(n³) so not a production-scale forecaster.

In [ ]:
frame = simulate_solar_output(SolarSimulationConfig(n_days=4, seed=7))
train = frame.iloc[:72]
test = frame.iloc[72:84]
forecast = fit_gp_solar(train, test, n_iter=40)
print(f"Train={len(train)} h  Test={len(test)} h  Peak={frame['solar_mw'].max():.0f} MW")

In [ ]:
plot_df = pd.concat(
    [
        train.assign(split="train"),
        test.assign(split="test"),
    ],
    ignore_index=True,
)
t_all = plot_df["valid_time"]
t_test = test["valid_time"]
mean = forecast.mean
std = forecast.std

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(t_all, plot_df["solar_mw"], color=COLOR_ACTUAL, linewidth=0.9, label="Actual")
ax.axvline(test["valid_time"].iloc[0], color="0.5", linestyle=":", linewidth=1.2, label="Forecast origin")
ax.plot(t_test, mean, color=COLOR_GP_MEAN, linewidth=1.2, label="GP mean")
ax.fill_between(t_test, mean - 2 * std, mean + 2 * std, alpha=0.25, color=COLOR_GP_MEAN, label="± 2σ (95% credible)")
ax.set_ylabel("Solar output (MW)")
ax.set_title("ExactGP: RBF + periodic — 12 h horizon")
ax.legend(loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()
save_summary_figure(fig, "nb03_gp_solar_bands")
plt.close(fig)

## GP vs conformal (one paragraph)

These ± 2σ bands are **Bayesian credible intervals**: they say "if the RBF+periodic kernel and Gaussian noise model are right, ~95% of the posterior mass sits here." That honesty is **model-dependent** — misspecified kernels, cloud ramps, or fleet changes break it, and there is no finite-sample coverage guarantee on held-out hours. **CQR** (Notebook 2) wraps a quantile LightGBM you already train and delivers **marginal coverage** at a stated level from a calibration set, at the cost of wider intervals. ExactGP is also **O(n³)** (~100 points here); a wind desk with thousands of features and daily refits uses conformalized quantile models instead. Use GP to explain Bayesian UQ in interviews; use conformal for contract-grade forecast bands.